# Adversarial Robustness of Deep Learning Models for Time Series Classification
### CurAT-LSTM-FCN | Extension of Fawaz et al. (2019)
**Models:** ResNet · FCN · LSTM · LSTM-FCN Vanilla · CurAT-LSTM-FCN  
**Attacks:** FGSM · BIM · PGD · C&W (L2)  

---
### ★ HOW TO USE — CHANGE ONLY `DATASET_NAME` IN CELL 2 ★
1. Go to `Runtime → Change runtime type → GPU (T4)`
2. Upload your dataset `.txt` files to Drive folder: `My Drive/UCR_Datasets/DATASETNAME/`
3. In Cell 2 set `DATASET_NAME = "YourDataset"`
4. Run all cells top to bottom


### CELL 1 | Mount Google Drive

In [1]:
from google.colab import drive
import os

# Safe mount — skips if already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("✓ Drive mounted.")
else:
    print("✓ Drive already mounted — skipping.")


MessageError: Error: credential propagation was unsuccessful

### CELL 2 | Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import itertools, json, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models    import Model
from tensorflow.keras.layers    import (
    Input, Conv1D, BatchNormalization, Activation, Add,
    GlobalAveragePooling1D, Dense, LSTM, Dropout, Permute, concatenate
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils      import to_categorical
from tensorflow.keras.callbacks  import ReduceLROnPlateau, ModelCheckpoint
from sklearn.preprocessing import LabelEncoder
from sklearn.manifold      import MDS
from sklearn.metrics       import accuracy_score, confusion_matrix

matplotlib.rcParams['figure.dpi']     = 150
matplotlib.rcParams['savefig.dpi']    = 300
matplotlib.rcParams['font.size']      = 12
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['axes.labelsize'] = 12
matplotlib.rcParams['legend.fontsize']= 10

print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))


### CELL 3 | Attack Functions (FGSM · BIM · PGD · C&W)

In [ ]:
def fgsm_attack(model, X, Y_true, epsilon, batch_size=None):
    if batch_size is None: batch_size = ATK_BATCH
    X_adv = np.zeros_like(X)
    for s in range(0, len(X), batch_size):
        e  = min(s + batch_size, len(X))
        Xb = tf.Variable(tf.cast(X[s:e], tf.float32))
        Yb = tf.cast(Y_true[s:e], tf.float32)
        with tf.GradientTape() as tape:
            loss = tf.keras.losses.categorical_crossentropy(Yb, model(Xb, training=False))
        X_adv[s:e] = (Xb + epsilon * tf.sign(tape.gradient(loss, Xb))).numpy()
    return X_adv

def bim_attack(model, X, Y_true, epsilon, alpha=None, num_iter=100, batch_size=None):
    if batch_size is None: batch_size = ATK_BATCH
    if alpha is None: alpha = epsilon / 10.0
    X_adv = X.copy().astype(np.float32)
    for s in range(0, len(X), batch_size):
        e  = min(s + batch_size, len(X))
        Xb = X[s:e].astype(np.float32)
        Yb = tf.cast(Y_true[s:e], tf.float32)
        Xa = Xb.copy()
        for _ in range(num_iter):
            Xt = tf.Variable(tf.cast(Xa, tf.float32))
            with tf.GradientTape() as tape:
                loss = tf.keras.losses.categorical_crossentropy(Yb, model(Xt, training=False))
            Xa = Xa + alpha * tf.sign(tape.gradient(loss, Xt)).numpy()
            Xa = np.clip(Xa, Xb - epsilon, Xb + epsilon)
        X_adv[s:e] = Xa
    return X_adv

def pgd_attack(model, X, Y_true, epsilon, alpha=None, num_iter=100, batch_size=None):
    if batch_size is None: batch_size = ATK_BATCH
    if alpha is None: alpha = epsilon / 10.0
    X_best = X.copy().astype(np.float32)
    for s in range(0, len(X), batch_size):
        e    = min(s + batch_size, len(X))
        Xb   = X[s:e].astype(np.float32)
        Yb   = tf.cast(Y_true[s:e], tf.float32)
        lb   = np.full(len(Xb), -np.inf, dtype=np.float32)
        Xbst = Xb.copy()
        noise = np.random.uniform(-epsilon, epsilon, Xb.shape).astype(np.float32)
        Xa    = np.clip(Xb + noise, Xb - epsilon, Xb + epsilon)
        for _ in range(num_iter):
            Xt = tf.Variable(tf.cast(Xa, tf.float32))
            with tf.GradientTape() as tape:
                loss = tf.keras.losses.categorical_crossentropy(Yb, model(Xt, training=False))
            Xa = Xa + alpha * tf.sign(tape.gradient(loss, Xt)).numpy()
            Xa = np.clip(Xa, Xb - epsilon, Xb + epsilon)
        lc = tf.keras.losses.categorical_crossentropy(
             Yb, model(tf.cast(Xa, tf.float32), training=False)).numpy()
        imp = lc > lb; Xbst[imp] = Xa[imp]
        X_best[s:e] = Xbst
    return X_best

def build_logit_model(model):
    gap_out  = model.layers[-2].output
    logit_out= Dense(model.layers[-1].units, use_bias=True, name='logits_out')(gap_out)
    lm = Model(inputs=model.input, outputs=logit_out)
    lm.get_layer('logits_out').set_weights(model.layers[-1].get_weights())
    return lm

def cw_l2_attack(model, X, Y_true, c=1.0, kappa=0.0,
                 num_iter=200, lr=0.01, batch_size=64):
    lm      = build_logit_model(model)
    X_adv   = np.zeros_like(X)
    n_batch = int(np.ceil(len(X) / batch_size))
    for b in range(n_batch):
        Xb  = X[b*batch_size:(b+1)*batch_size].astype(np.float32)
        Yb  = Y_true[b*batch_size:(b+1)*batch_size].astype(np.float32)
        n   = len(Xb)
        delta = tf.Variable(tf.zeros_like(Xb), trainable=True)
        opt   = tf.keras.optimizers.Adam(learning_rate=lr)
        for _ in range(num_iter):
            with tf.GradientTape() as tape:
                Xa    = tf.constant(Xb) + delta
                logit = lm(Xa, training=False)
                tl    = tf.reduce_sum(logit * tf.constant(Yb), axis=1)
                ol    = tf.reduce_max(logit - 1e9 * tf.constant(Yb), axis=1)
                fl    = tf.maximum(tl - ol + kappa, tf.zeros(n))
                l2    = tf.reduce_sum(tf.square(delta), axis=[1,2])
                loss  = tf.reduce_mean(l2 + c * fl)
            opt.apply_gradients(zip(tape.gradient(loss, [delta]), [delta]))
        X_adv[b*batch_size:(b+1)*batch_size] = (Xb + delta.numpy())
        if (b+1) % 20 == 0:
            print(f"  C&W: batch {b+1}/{n_batch}")
    return X_adv.astype(np.float32)

print("✓ All attack functions defined.")


### CELL 4 | Training Helper Functions + Config

In [ ]:
# ============================================================================
# CELL 7 | Training Helper Functions + Config  (seed-aware version)
# ============================================================================

WEIGHTS = {
    'resnet'  : '/content/best_resnet.weights.h5',
    'fcn'     : '/content/best_fcn.weights.h5',
    'lstm'    : '/content/best_lstm.weights.h5',
    'vanilla' : '/content/best_lstmfcn_vanilla.weights.h5',
    'robust'  : '/content/best_lstmfcn_robust.weights.h5',
}

import random

def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception as e:
        print("enable_op_determinism not available:", e)


def smooth_curve(v, w=0.85):
    s, last = [], v[0]
    for x in v: sv = last*w + x*(1-w); s.append(sv); last = sv
    return s


def save_history(h, name):
    path = f"{CKPT_DIR}/{name}_history.json"
    with open(path, 'w') as f: json.dump(h, f)
    print(f"  [Drive] History saved: {name}_history.json")


def load_history(name):
    path = f"{CKPT_DIR}/{name}_history.json"
    if os.path.exists(path):
        with open(path) as f: return json.load(f)
    return None


def dummy_hist():
    return {'loss':[0],'val_loss':[0],'accuracy':[0],'val_accuracy':[0]}


def train_standard(model, name, label, seed=None):
    # seed=None keeps ORIGINAL behavior/paths unchanged — use this for
    # ResNet, FCN, LSTM (trained once, not part of the multi-seed check).
    # seed=0..4 is for Vanilla and CurAT only, during the multi-seed re-run.
    suffix  = f"_seed{seed}" if seed is not None else ""
    drive_w = f"{CKPT_DIR}/{name}{suffix}.weights.h5"
    local_w = f"/content/best_{name}{suffix}.weights.h5"

    if os.path.exists(drive_w):
        print(f"✓ {label} (seed={seed}) weights found on Drive — loading.")
        model.load_weights(drive_w)
        h = load_history(f"{name}{suffix}") or dummy_hist()
        print(f"  Best val acc: {max(h['val_accuracy'])*100:.2f}%")
        return h

    print("="*60)
    print(f"  Training {label} (seed={seed})")
    print(f"  Epochs: {EPOCHS}  |  Saving to Drive on completion")
    print("="*60)
    cbs = [
        ReduceLROnPlateau(monitor='loss', factor=0.5,
                          patience=LR_PATIENCE, min_lr=1e-6, verbose=0),
        ModelCheckpoint(local_w, monitor='val_accuracy',
                        save_best_only=True, save_weights_only=True, verbose=0)
    ]
    hist = model.fit(X_train, y_train_cat,
                     epochs=EPOCHS, batch_size=BATCH_SIZE,
                     validation_data=(X_test, y_test_cat),
                     callbacks=cbs, verbose=1)
    model.load_weights(local_w)
    model.save_weights(drive_w)
    h = {k:[float(v) for v in vals] for k,vals in hist.history.items()}
    save_history(h, f"{name}{suffix}")
    print(f"\n  ✓ {label} (seed={seed}) done. Best val acc = {max(h['val_accuracy'])*100:.2f}%")
    print(f"  Weights + history saved to Drive.")
    return h


print("✓ Training helpers ready (seed-aware).")

### Cell 5 — function mein convert

In [ ]:
def load_dataset(DATASET_NAME):
    TRAIN_PATH = f"/content/drive/MyDrive/{DATASET_NAME}_TRAIN.txt"
    TEST_PATH  = f"/content/drive/MyDrive/{DATASET_NAME}_TEST.txt"

    def load_ucr_raw(path):
        df = pd.read_csv(path, sep=r'\s+', header=None, engine='python')
        return df.iloc[:,1:].values.astype(np.float32), df.iloc[:,0].values

    X_train_raw, y_train_raw = load_ucr_raw(TRAIN_PATH)
    X_test_raw,  y_test_raw  = load_ucr_raw(TEST_PATH)

    n_train = X_train_raw.shape[0]
    assert n_train >= 400, f"Only {n_train} training samples — minimum is 400."

    le = LabelEncoder()
    le.fit(np.concatenate([y_train_raw, y_test_raw]))
    y_train_int = le.transform(y_train_raw)
    y_test_int  = le.transform(y_test_raw)
    num_classes   = len(le.classes_)
    series_length = X_train_raw.shape[1]

    X_train = X_train_raw[:, :, np.newaxis]
    X_test  = X_test_raw[:,  :, np.newaxis]
    X_train = (X_train - X_train.mean(axis=1, keepdims=True)) / (X_train.std(axis=1, keepdims=True) + 1e-8)
    X_test  = (X_test  - X_test.mean(axis=1,  keepdims=True)) / (X_test.std(axis=1,  keepdims=True) + 1e-8)

    y_train_cat = to_categorical(y_train_int, num_classes)
    y_test_cat  = to_categorical(y_test_int,  num_classes)

    input_shape = (series_length, 1)
    EPSILON     = 0.1
    ATK_BATCH   = 64 if series_length > 1000 else (128 if series_length > 500 else 256)

    if n_train < 1000:
        EPOCHS, LR_PATIENCE = 1500, 100
    elif n_train < 3000:
        EPOCHS, LR_PATIENCE = 1000, 75
    else:
        EPOCHS, LR_PATIENCE = 500, 50
    BATCH_SIZE = 64

    print(f"Dataset       : {DATASET_NAME}")
    print(f"Train shape   : {X_train.shape}  |  Test shape: {X_test.shape}")
    print(f"Classes       : {num_classes} → {le.classes_.tolist()}")
    print(f"Series length : {series_length}")
    print(f"Epochs        : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  ε: {EPSILON}")

    return {
        'X_train': X_train, 'y_train_cat': y_train_cat, 'y_train_int': y_train_int,
        'X_test': X_test, 'y_test_cat': y_test_cat, 'y_test_int': y_test_int,
        'num_classes': num_classes, 'series_length': series_length,
        'input_shape': input_shape, 'EPSILON': EPSILON, 'ATK_BATCH': ATK_BATCH,
        'EPOCHS': EPOCHS, 'LR_PATIENCE': LR_PATIENCE, 'BATCH_SIZE': BATCH_SIZE,
    }

 Cell 6: build_fixed_models function

In [ ]:
def build_fixed_models(input_shape, num_classes):
    def res_block(x, f):
        c = Conv1D(f, 8, padding='same', use_bias=False)(x)
        c = BatchNormalization()(c); c = Activation('relu')(c)
        c = Conv1D(f, 5, padding='same', use_bias=False)(c)
        c = BatchNormalization()(c); c = Activation('relu')(c)
        c = Conv1D(f, 3, padding='same', use_bias=False)(c)
        c = BatchNormalization()(c)
        s = Conv1D(f, 1, padding='same', use_bias=False)(x)
        s = BatchNormalization()(s)
        return Activation('relu')(Add()([s, c]))

    def build_resnet():
        inp = Input(shape=input_shape)
        x   = res_block(inp, 64); x = res_block(x, 128); x = res_block(x, 128)
        x   = GlobalAveragePooling1D()(x)
        return Model(inp, Dense(num_classes, activation='softmax')(x), name='ResNet')

    def build_fcn():
        inp = Input(shape=input_shape)
        x   = Conv1D(128, 8, padding='same', use_bias=False)(inp)
        x   = BatchNormalization()(x); x = Activation('relu')(x)
        x   = Conv1D(256, 5, padding='same', use_bias=False)(x)
        x   = BatchNormalization()(x); x = Activation('relu')(x)
        x   = Conv1D(128, 3, padding='same', use_bias=False)(x)
        x   = BatchNormalization()(x); x = Activation('relu')(x)
        x   = GlobalAveragePooling1D()(x)
        return Model(inp, Dense(num_classes, activation='softmax')(x), name='FCN')

    def build_lstm():
        inp = Input(shape=input_shape)
        x   = LSTM(128, return_sequences=True)(inp)
        x   = Dropout(0.5)(x)
        x   = LSTM(64, return_sequences=False)(x)
        x   = Dropout(0.3)(x)
        return Model(inp, Dense(num_classes, activation='softmax')(x), name='LSTM')

    resnet_model = build_resnet()
    fcn_model    = build_fcn()
    lstm_model   = build_lstm()

    for m in [resnet_model, fcn_model, lstm_model]:
        m.compile(Adam(1e-3), 'categorical_crossentropy', metrics=['accuracy'])

    print("✓ ResNet, FCN, LSTM built:")
    for m in [resnet_model, fcn_model, lstm_model]:
        print(f"  {m.name:<25} params={m.count_params():,}")

    return resnet_model, fcn_model, lstm_model

### Cell : Build SEED Models

In [ ]:
def build_seed_models(input_shape, num_classes, seed):
    set_all_seeds(seed)

    def build_lstm_fcn(name):
        inp      = Input(shape=input_shape)
        x        = Conv1D(128, 8, padding='same', use_bias=False)(inp)
        x        = BatchNormalization()(x); x = Activation('relu')(x)
        x        = Conv1D(256, 5, padding='same', use_bias=False)(x)
        x        = BatchNormalization()(x); x = Activation('relu')(x)
        x        = Conv1D(128, 3, padding='same', use_bias=False)(x)
        x        = BatchNormalization()(x); x = Activation('relu')(x)
        fcn_out  = GlobalAveragePooling1D()(x)
        lstm_in  = Permute((2, 1))(inp)
        lstm_out = LSTM(8)(lstm_in)
        lstm_out = Dropout(0.8)(lstm_out)
        merged   = concatenate([fcn_out, lstm_out])
        return Model(inp, Dense(num_classes, activation='softmax')(merged), name=name)

    lstmfcn_vanilla = build_lstm_fcn('LSTM_FCN_Vanilla')
    lstmfcn_vanilla.compile(Adam(1e-3), 'categorical_crossentropy', metrics=['accuracy'])

    lstmfcn_robust = build_lstm_fcn('LSTM_FCN_Robust')
    lstmfcn_robust.compile(Adam(1e-3), 'categorical_crossentropy', metrics=['accuracy'])

    return lstmfcn_vanilla, lstmfcn_robust

### CELL 8: Generate_or_load_attacks

In [ ]:
def generate_or_load_attacks(resnet_model, X_test, y_test_cat, EPSILON, series_length, CKPT_DIR):
    ATTACK_GEN_SEED = 999
    adv_dir = f"{CKPT_DIR}/adv_arrays"
    os.makedirs(adv_dir, exist_ok=True)

    paths = {
        'fgsm': f"{adv_dir}/X_fgsm.npy",
        'bim':  f"{adv_dir}/X_bim.npy",
        'pgd':  f"{adv_dir}/X_pgd.npy",
        'cw':   f"{adv_dir}/X_cw.npy",
    }

    if all(os.path.exists(p) for p in paths.values()):
        print(f"✓ Adversarial arrays already exist — loading, NOT regenerating.")
        X_fgsm = np.load(paths['fgsm']); X_bim = np.load(paths['bim'])
        X_pgd  = np.load(paths['pgd']);  X_cw  = np.load(paths['cw'])
    else:
        set_all_seeds(ATTACK_GEN_SEED)
        print(f"Generating adversarial examples using ResNet (ε={EPSILON})...")

        print("  FGSM...", end=" ")
        X_fgsm = fgsm_attack(resnet_model, X_test, y_test_cat, epsilon=EPSILON)
        print("done")

        print("  BIM (100 iter)...", end=" ")
        X_bim = bim_attack(resnet_model, X_test, y_test_cat, epsilon=EPSILON, num_iter=100)
        print("done")

        print("  PGD (100 iter)...", end=" ")
        X_pgd = pgd_attack(resnet_model, X_test, y_test_cat, epsilon=EPSILON, num_iter=100)
        print("done")

        cw_bs = 32 if series_length > 1000 else 64
        print(f"  C&W (200 iter, batch={cw_bs})...")
        X_cw = cw_l2_attack(resnet_model, X_test, y_test_cat,
                            c=1.0, kappa=0.0, num_iter=200, lr=0.01, batch_size=cw_bs)
        print("  C&W done")

        np.save(paths['fgsm'], X_fgsm); np.save(paths['bim'], X_bim)
        np.save(paths['pgd'], X_pgd);   np.save(paths['cw'], X_cw)
        print(f"✓ All 4 adversarial arrays saved: {adv_dir}")

    print("\nPerturbation norms:")
    for name, Xa in [('FGSM',X_fgsm),('BIM',X_bim),('PGD',X_pgd),('C&W',X_cw)]:
        d = Xa - X_test
        print(f"  {name:<6}: L∞={np.max(np.abs(d)):.4f}  L2(avg)={np.mean(np.sqrt(np.sum(d**2,axis=(1,2)))):.4f}")

    return X_fgsm, X_bim, X_pgd, X_cw

### CELL 9: Train_CurAT

In [ ]:
def train_curat(seed, DATASET_NAME, CKPT_DIR, lstmfcn_vanilla, lstmfcn_robust,
                 X_train, y_train_cat, X_test, y_test_cat, y_test_int,
                 series_length, num_classes, EPOCHS, LR_PATIENCE, BATCH_SIZE, ATK_BATCH):

    SKIP_ROBUST_TRAINING = False

    ADV_EPS = min(0.1, max(0.01, 8.0 / np.sqrt(series_length)))
    BIM_TRAIN_ITER = int(np.clip(np.round(5 + 5 * (series_length - 200) / 1600), 5, 10))
    BIM_ALPHA      = ADV_EPS / BIM_TRAIN_ITER
    difficulty = float(np.log1p(num_classes) * np.sqrt(series_length) / np.sqrt(len(X_train)))
    MAX_ADV    = float(np.clip(0.70 - 0.15 * np.log1p(difficulty), 0.35, 0.70))
    WARMUP = min(750, int(EPOCHS * 0.50))
    RAMP   = min(900, int(EPOCHS * 0.75))

    drive_robust_w = f"{CKPT_DIR}/robust_fixed_seed{seed}.weights.h5"
    van_w          = f"{CKPT_DIR}/vanilla_seed{seed}.weights.h5"
    local_robust_w = f"/content/best_robust_seed{seed}.weights.h5"

    print(f"Dataset : {DATASET_NAME}  |  T={series_length}  |  C={num_classes}  |  "
          f"n_train={len(X_train)}  |  seed={seed}")
    print(f"ε={ADV_EPS:.4f}  |  BIM_iter={BIM_TRAIN_ITER}  |  MAX_ADV={MAX_ADV:.2f}  |  difficulty={difficulty:.3f}")
    print(f"Phase1(clean): 0→{WARMUP}  |  Phase2(ramp): {WARMUP}→{RAMP}  |  Phase3(FGSM+BIM): {RAMP}→{EPOCHS}")

    def adv_val_acc_fast(model, eps):
        preds = np.zeros(len(X_test), dtype=np.int32)
        for s in range(0, len(X_test), ATK_BATCH):
            e  = min(s + ATK_BATCH, len(X_test))
            Xv = tf.Variable(tf.cast(X_test[s:e], tf.float32))
            Yv = tf.cast(y_test_cat[s:e], tf.float32)
            with tf.GradientTape() as tape:
                loss = tf.keras.losses.categorical_crossentropy(Yv, model(Xv, training=False))
            Xf = (Xv + eps * tf.sign(tape.gradient(loss, Xv))).numpy()
            preds[s:e] = np.argmax(model.predict(Xf, verbose=0), axis=1)
        return float(np.mean(preds == y_test_int))

    def real_clean_acc(model):
        return float(np.mean(np.argmax(model.predict(X_test, verbose=0), axis=1) == y_test_int))

    if os.path.exists(drive_robust_w):
        print(f"✓ CurAT weights already exist for seed={seed} — loading, NOT retraining.")
        lstmfcn_robust.load_weights(drive_robust_w)
        rc, ra = real_clean_acc(lstmfcn_robust), adv_val_acc_fast(lstmfcn_robust, ADV_EPS)
        print(f"  Real clean={rc*100:.2f}%  Real FGSM(ε)={ra*100:.2f}%")
        hist_robust = load_history(f'{DATASET_NAME}_robust_fixed_seed{seed}') or {
            'loss': [], 'val_accuracy': [], 'val_acc': [],
            'saved_checkpoint_clean_acc': rc, 'saved_checkpoint_adv_acc': ra
        }
        return lstmfcn_robust, hist_robust

    from tensorflow.keras.callbacks import Callback

    class CollapseGuard(Callback):
        def __init__(self, n_cls, warmup, patience=150):
            super().__init__()
            self.threshold = (1.0/n_cls)*1.5; self.warmup=warmup; self.patience=patience; self.wait=0
        def on_epoch_end(self, epoch, logs=None):
            va = logs.get('val_accuracy', 1.0)
            if epoch >= self.warmup and va < self.threshold:
                self.wait += 1
                if self.wait >= self.patience:
                    print(f"\n⚠ CollapseGuard: stopped epoch {epoch+1}")
                    self.model.stop_training = True
            else:
                self.wait = 0

    if os.path.exists(van_w):
        lstmfcn_vanilla.load_weights(van_w)
        lstmfcn_robust.set_weights(lstmfcn_vanilla.get_weights())
        print(f"✓ Warm-start: Vanilla (seed={seed}) → CurAT (seed={seed})")
    else:
        print(f"⚠  Vanilla weights for seed={seed} nahi mile — train Vanilla first")

    set_all_seeds(seed)

    optimizer  = tf.keras.optimizers.Adam(1e-3)
    loss_fn    = tf.keras.losses.CategoricalCrossentropy()
    best_clean = 0.0; best_adv = 0.0
    best_clean_epoch = -1; best_adv_epoch = -1; best_adv_clean_at_save = 0.0
    lr_wait = 0; prev_loss = np.inf; current_lr = 1e-3
    n_samples = len(X_train); use_bim = False
    adv_log = {'loss': [], 'val_accuracy': [], 'val_acc': []}
    guard = CollapseGuard(num_classes, WARMUP)
    guard.set_model(lstmfcn_robust)

    print("\nTraining started...\n")

    for epoch in range(EPOCHS):
        if epoch < WARMUP:
            adv_ratio, phase_name = 0.0, "Phase1-Clean"
        elif epoch < RAMP:
            adv_ratio = ((epoch - WARMUP) / (RAMP - WARMUP)) * MAX_ADV
            phase_name = "Phase2-Ramp"
        else:
            adv_ratio, phase_name = MAX_ADV, "Phase3-MultiAtk"

        perm = np.random.permutation(n_samples)
        Xs, Ys = X_train[perm], y_train_cat[perm]
        epoch_losses = []

        for s in range(0, n_samples, BATCH_SIZE):
            e = min(s + BATCH_SIZE, n_samples)
            Xb = Xs[s:e].astype(np.float32); Yb = Ys[s:e].astype(np.float32)

            if adv_ratio > 0:
                n_adv = max(1, int(len(Xb) * adv_ratio))
                Xb_orig = Xb[:n_adv]; Yb_adv = Yb[:n_adv]

                if phase_name == "Phase3-MultiAtk" and use_bim:
                    Xa = Xb_orig.copy()
                    for _ in range(BIM_TRAIN_ITER):
                        Xv = tf.Variable(tf.cast(Xa, tf.float32))
                        with tf.GradientTape() as t2:
                            la = tf.keras.losses.categorical_crossentropy(
                                tf.cast(Yb_adv, tf.float32), lstmfcn_robust(Xv, training=False))
                        Xa = Xa + BIM_ALPHA * tf.sign(t2.gradient(la, Xv)).numpy()
                        Xa = np.clip(Xa, Xb_orig - ADV_EPS, Xb_orig + ADV_EPS)
                    Xadv = Xa
                else:
                    Xv = tf.Variable(tf.cast(Xb_orig, tf.float32))
                    with tf.GradientTape() as t2:
                        la = tf.keras.losses.categorical_crossentropy(
                            tf.cast(Yb_adv, tf.float32), lstmfcn_robust(Xv, training=False))
                    Xadv = (Xv + ADV_EPS * tf.sign(t2.gradient(la, Xv))).numpy()

                Xb = np.concatenate([Xadv, Xb[n_adv:]], axis=0)

            with tf.GradientTape() as tape:
                preds = lstmfcn_robust(tf.cast(Xb, tf.float32), training=True)
                loss = loss_fn(tf.cast(Yb, tf.float32), preds)
            grads = tape.gradient(loss, lstmfcn_robust.trainable_variables)
            grads, _ = tf.clip_by_global_norm(grads, clip_norm=1.0)
            optimizer.apply_gradients(zip(grads, lstmfcn_robust.trainable_variables))
            epoch_losses.append(float(loss))

        if phase_name == "Phase3-MultiAtk":
            use_bim = not use_bim

        ml = float(np.mean(epoch_losses))
        va = float(np.mean(np.argmax(lstmfcn_robust.predict(X_test, verbose=0), axis=1) == y_test_int))
        adv_log['loss'].append(ml); adv_log['val_accuracy'].append(va); adv_log['val_acc'].append(va)

        if epoch < WARMUP:
            if va > best_clean:
                best_clean, best_clean_epoch = va, epoch
                lstmfcn_robust.save_weights(local_robust_w)
        else:
            adv_va = adv_val_acc_fast(lstmfcn_robust, ADV_EPS)
            if adv_va > best_adv:
                best_adv, best_adv_epoch, best_adv_clean_at_save = adv_va, epoch, va
                lstmfcn_robust.save_weights(local_robust_w)

        if ml < prev_loss - 1e-5:
            lr_wait = 0; prev_loss = ml
        else:
            lr_wait += 1
            if lr_wait >= LR_PATIENCE and current_lr > 1e-6:
                current_lr = max(current_lr*0.5, 1e-6)
                optimizer.learning_rate.assign(current_lr); lr_wait = 0; prev_loss = ml

        if (epoch+1) % 100 == 0 or epoch == 0:
            atk = "BIM-"+str(BIM_TRAIN_ITER) if (use_bim and phase_name=="Phase3-MultiAtk") else "FGSM"
            print(f"  Ep {epoch+1:>5}/{EPOCHS}  loss={ml:.4f}  val_acc={va*100:.2f}%  "
                  f"mix={adv_ratio:.0%}  atk={atk}  lr={current_lr:.1e}  [{phase_name}]")

        guard.on_epoch_end(epoch, {'val_accuracy': va})
        if lstmfcn_robust.stop_training:
            break

    lstmfcn_robust.load_weights(local_robust_w)
    lstmfcn_robust.save_weights(drive_robust_w)
    saved_clean_acc = best_adv_clean_at_save if best_adv_epoch > best_clean_epoch else best_clean
    saved_epoch = best_adv_epoch if best_adv_epoch > best_clean_epoch else best_clean_epoch
    adv_log['saved_checkpoint_clean_acc'] = saved_clean_acc
    adv_log['saved_checkpoint_adv_acc']   = best_adv
    adv_log['saved_checkpoint_epoch']     = saved_epoch
    save_history(adv_log, f'{DATASET_NAME}_robust_fixed_seed{seed}')

    print(f"\n✓ Done (seed={seed}).  Saved checkpoint epoch: {saved_epoch}")
    print(f"  Real clean={saved_clean_acc*100:.2f}%  Real adv(FGSM,ε)={best_adv*100:.2f}%")
    print(f"  Saved → {drive_robust_w}")

    return lstmfcn_robust, adv_log

### CELL 10: Evaluate_dataset_seed

In [ ]:
def evaluate_dataset_seed(DATASET_NAME, seed, CKPT_DIR,
                           resnet_model, fcn_model, lstm_model,
                           lstmfcn_vanilla, lstmfcn_robust,
                           X_test, y_test_int, X_fgsm, X_bim, X_pgd, X_cw,
                           multiseed_records):

    all_models = {
        'ResNet'            : resnet_model,
        'FCN'               : fcn_model,
        'LSTM'              : lstm_model,
        'LSTM-FCN Vanilla'  : lstmfcn_vanilla,
        'CurAT-LSTM-FCN'    : lstmfcn_robust,
    }
    attack_mode = {
        'ResNet'           : 'White-box',
        'FCN'              : 'Black-box',
        'LSTM'             : 'Black-box',
        'LSTM-FCN Vanilla' : 'Black-box',
        'CurAT-LSTM-FCN'   : 'Black-box',
    }
    model_names = list(all_models.keys())

    FIXED_MODELS = ['ResNet', 'FCN', 'LSTM']
    already_logged_fixed = any(
        r['dataset'] == DATASET_NAME and r['model'] in FIXED_MODELS
        for r in multiseed_records
    )
    models_to_eval = {
        mname: model for mname, model in all_models.items()
        if mname not in FIXED_MODELS or not already_logged_fixed
    }

    print(f"Dataset={DATASET_NAME}  seed={seed}  |  Evaluating {len(models_to_eval)} model(s)...")

    for mname, model in models_to_eval.items():
        preds = {
            'clean': np.argmax(model.predict(X_test, verbose=0), axis=1),
            'fgsm' : np.argmax(model.predict(X_fgsm, verbose=0), axis=1),
            'bim'  : np.argmax(model.predict(X_bim,  verbose=0), axis=1),
            'pgd'  : np.argmax(model.predict(X_pgd,  verbose=0), axis=1),
            'cw'   : np.argmax(model.predict(X_cw,   verbose=0), axis=1),
        }
        ac,fgsm,bim,pgd,cw = [accuracy_score(y_test_int, preds[k]) for k in ['clean','fgsm','bim','pgd','cw']]

        logged_seed = None if mname in FIXED_MODELS else seed
        multiseed_records.append({
            'dataset': DATASET_NAME, 'model': mname, 'seed': logged_seed,
            'clean': ac, 'fgsm': fgsm, 'bim': bim, 'pgd': pgd, 'cw': cw,
            'mean_attack': np.mean([fgsm, bim, pgd, cw])
        })
        print(f"  ✓ {mname:<25} Clean={ac*100:.2f}%")

    snapshot_path = f"{CKPT_DIR}/{DATASET_NAME}_multiseed_records.xlsx"
    pd.DataFrame(multiseed_records).to_excel(snapshot_path, index=False)
    print(f"✓ Snapshot saved: {snapshot_path}")

    return multiseed_records

### CELL 11: Master Loop

In [ ]:
# ══════════════════════════════════════════════════════════
# MASTER LOOP | 15 datasets × 5 seeds, fully automated
# ══════════════════════════════════════════════════════════

DATASETS_15 = [
    "ECG5000"
]
SEEDS = [0, 1, 2, 3, 4]

for DATASET_NAME in DATASETS_15:
    print(f"\n{'='*70}\n  STARTING: {DATASET_NAME}\n{'='*70}")

    TRAIN_PATH = f"/content/drive/MyDrive/{DATASET_NAME}_TRAIN.txt"
    TEST_PATH  = f"/content/drive/MyDrive/{DATASET_NAME}_TEST.txt"
    CKPT_DIR   = f"/content/drive/MyDrive/UCR_Datasets/{DATASET_NAME}_checkpoints"
    SAVE_DIR   = f"/content/results/{DATASET_NAME}"
    os.makedirs(CKPT_DIR, exist_ok=True)
    os.makedirs(SAVE_DIR, exist_ok=True)

    if not (os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH)):
        print(f"⚠ SKIPPING {DATASET_NAME} — TRAIN/TEST file not found on Drive")
        continue

    # ── load dataset, inject into globals (train_standard needs these as globals) ──
    data = load_dataset(DATASET_NAME)
    globals().update(data)
    WEIGHTS = {
        'resnet' : f"/content/best_{DATASET_NAME}_resnet.weights.h5",
        'fcn'    : f"/content/best_{DATASET_NAME}_fcn.weights.h5",
        'lstm'   : f"/content/best_{DATASET_NAME}_lstm.weights.h5",
    }

    resnet_model, fcn_model, lstm_model = build_fixed_models(input_shape, num_classes)

    hist_resnet = train_standard(resnet_model, 'resnet', 'ResNet (white-box source)')
    hist_fcn    = train_standard(fcn_model,    'fcn',    'FCN (black-box baseline)')
    hist_lstm   = train_standard(lstm_model,   'lstm',   'LSTM (black-box baseline)')

    X_fgsm, X_bim, X_pgd, X_cw = generate_or_load_attacks(
        resnet_model, X_test, y_test_cat, EPSILON, series_length, CKPT_DIR)

    snapshot_path = f"{CKPT_DIR}/{DATASET_NAME}_multiseed_records.xlsx"
    if os.path.exists(snapshot_path):
        multiseed_records = pd.read_excel(snapshot_path).to_dict('records')
        print(f"✓ Resumed {len(multiseed_records)} prior records for {DATASET_NAME}")
    else:
        multiseed_records = []

    for seed in SEEDS:
        print(f"\n--- {DATASET_NAME} | seed={seed} ---")

        lstmfcn_vanilla, lstmfcn_robust = build_seed_models(input_shape, num_classes, seed)

        hist_vanilla = train_standard(
            lstmfcn_vanilla, 'vanilla', 'LSTM-FCN Vanilla (black-box baseline)', seed=seed)

        lstmfcn_robust, hist_robust = train_curat(
            seed, DATASET_NAME, CKPT_DIR, lstmfcn_vanilla, lstmfcn_robust,
            X_train, y_train_cat, X_test, y_test_cat, y_test_int,
            series_length, num_classes, EPOCHS, LR_PATIENCE, BATCH_SIZE, ATK_BATCH)

        multiseed_records = evaluate_dataset_seed(
            DATASET_NAME, seed, CKPT_DIR, resnet_model, fcn_model, lstm_model,
            lstmfcn_vanilla, lstmfcn_robust, X_test, y_test_int,
            X_fgsm, X_bim, X_pgd, X_cw, multiseed_records)

        # ── safety-check: flag suspicious 0%/100%/NaN values immediately ──
        last = [r for r in multiseed_records
                if r['dataset']==DATASET_NAME and r['model']=='CurAT-LSTM-FCN' and r['seed']==seed]
        if last:
            row = last[-1]
            for k in ['clean','fgsm','bim','pgd','cw']:
                v = row[k]
                if v == 0.0 or v == 1.0 or (isinstance(v,float) and np.isnan(v)):
                    print(f"  ⚠⚠⚠ SANITY FLAG: {DATASET_NAME} seed={seed} CurAT {k}={v} — verify manually!")

    print(f"\n✓✓✓ {DATASET_NAME} COMPLETE (all 5 seeds) ✓✓✓")

print("\n\n████ ALL 15 DATASETS COMPLETE ████")

In [ ]:
# ── after the master loop, expose these for the display cells (Cell 16-23) ──
all_models = {'ResNet':resnet_model,'FCN':fcn_model,'LSTM':lstm_model,
              'LSTM-FCN Vanilla':lstmfcn_vanilla,'CurAT-LSTM-FCN':lstmfcn_robust}
attack_mode = {'ResNet':'White-box','FCN':'Black-box','LSTM':'Black-box',
               'LSTM-FCN Vanilla':'Black-box','CurAT-LSTM-FCN':'Black-box'}
model_names = list(all_models.keys())
FIXED_MODELS = ['ResNet','FCN','LSTM']

### CELL 16 | Results Table (Console)

In [ ]:
def get_row(mname):
    target_seed = None if mname in FIXED_MODELS else seed
    match = [r for r in multiseed_records
             if r['dataset']==DATASET_NAME and r['model']==mname and r['seed']==target_seed]
    return match[-1] if match else None

SEP = "=" * 95
print(f"\n{SEP}")
print(f"  COMPLETE RESULTS — {DATASET_NAME}  (seed={seed})")
print(SEP)
print(f"  {'Model':<22} {'Clean':>8} {'FGSM':>8} {'BIM':>8} {'PGD':>8} {'C&W':>8}  "
      f"{'FGSM↓':>8} {'BIM↓':>8} {'PGD↓':>8} {'C&W↓':>8}  Mode")
print("  " + "-"*93)
for mname in model_names:
    row = get_row(mname)
    if row is None:
        print(f"  {mname:<22}  [no data — run Cell 15 for this seed first]")
        continue
    ac = row['clean']
    print(f"  {mname:<22} "
          f"{ac*100:>7.2f}% {row['fgsm']*100:>7.2f}% {row['bim']*100:>7.2f}% "
          f"{row['pgd']*100:>7.2f}% {row['cw']*100:>7.2f}%  "
          f"{(ac-row['fgsm'])*100:>7.2f}% {(ac-row['bim'])*100:>7.2f}% "
          f"{(ac-row['pgd'])*100:>7.2f}% {(ac-row['cw'])*100:>7.2f}%  {attack_mode[mname]}")
print(SEP)

### CELL 17 | Styled Results Table Figure

In [ ]:
row_colors = {'ResNet':'#d6eaf8','FCN':'#fef9e7','LSTM':'#f9ebea',
              'LSTM-FCN Vanilla':'#e9f7ef','CurAT-LSTM-FCN':'#fde8d8'}
col_labels = ['Model','Clean Acc','FGSM Acc','BIM Acc','PGD Acc','C&W Acc',
              'FGSM Drop↓','BIM Drop↓','PGD Drop↓','C&W Drop↓','Mode']
table_data, valid_models = [], []
for mname in model_names:
    row = get_row(mname)
    if row is None: continue
    valid_models.append(mname)
    ac = row['clean']
    table_data.append([mname, f"{ac*100:.2f}%"]
        + [f"{row[k]*100:.2f}%" for k in ['fgsm','bim','pgd','cw']]
        + [f"−{(ac-row[k])*100:.2f}%" for k in ['fgsm','bim','pgd','cw']]
        + [attack_mode[mname]])

fig, ax = plt.subplots(figsize=(24,4)); ax.axis('off')
tbl = ax.table(cellText=table_data, colLabels=col_labels, cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1.0,2.5)
for j in range(len(col_labels)):
    tbl[0,j].set_facecolor('#2c3e50'); tbl[0,j].set_text_props(color='white', fontweight='bold')
for i, mname in enumerate(valid_models):
    row = get_row(mname); ac = row['clean']
    for j in range(len(col_labels)):
        tbl[i+1,j].set_facecolor(row_colors[mname])
    for dk, akey in enumerate(['fgsm','bim','pgd','cw']):
        drop = ac - row[akey]; ci = 6+dk
        if drop > 0.15:
            tbl[i+1,ci].set_facecolor('#fadbd8'); tbl[i+1,ci].set_text_props(color='darkred', fontweight='bold')
        elif drop < 0.05:
            tbl[i+1,ci].set_facecolor('#d5f5e3'); tbl[i+1,ci].set_text_props(color='darkgreen', fontweight='bold')
ax.set_title(f"{DATASET_NAME} (seed={seed}) — Full Results Table (5 Models · 4 Attacks)",
             fontsize=11, fontweight='bold', pad=16)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/results_table_seed{seed}.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: results_table_seed{seed}.png")

### CELL 18 | Grouped Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(20, 6))
n_bars    = 5
x         = np.arange(len(model_names)) * (n_bars + 1.5)
width     = 0.65
bar_cols  = ['#2ecc71','#f39c12','#e74c3c','#9b59b6','#3498db']
bar_lbls  = ['Clean','FGSM','BIM','PGD','C&W']
bar_akeys = ['clean','fgsm','bim','pgd','cw']

rows = {m: get_row(m) for m in model_names}
missing = [m for m,r in rows.items() if r is None]
if missing:
    print(f"⚠ Skipping chart — no data yet for: {missing} (run Cell 15 for this seed)")
else:
    for i, (lbl, akey, color) in enumerate(zip(bar_lbls, bar_akeys, bar_cols)):
        vals = [rows[m][akey] for m in model_names]
        bars = ax.bar(x + i*width, vals, width, label=lbl, color=color, edgecolor='white', alpha=0.88)
        for bar, v in zip(bars, vals):
            ax.annotate(f'{v*100:.1f}%', xy=(bar.get_x()+bar.get_width()/2, v),
                        xytext=(0,3), textcoords='offset points',
                        ha='center', va='bottom', fontsize=6.5, rotation=90)

    ax.set_xticks(x + width*2); ax.set_xticklabels(model_names, fontsize=10)
    ax.set_ylabel("Test Accuracy", fontsize=12)
    ax.set_title(f"{DATASET_NAME} (seed={seed}) — Accuracy Under All Four Attacks  (ε={EPSILON})",
                 fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.22); ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
    for xi in x[1:]: ax.axvline(xi-0.8, color='gray', ls=':', lw=1.0)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/bar_all_attacks_seed{seed}.png", dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: bar_all_attacks_seed{seed}.png")

### CELL 19 | Accuracy Drop Heatmap

In [ ]:
rows = {m: get_row(m) for m in model_names}
missing = [m for m,r in rows.items() if r is None]
if missing:
    print(f"⚠ Skipping heatmap — no data yet for: {missing}")
else:
    drop_mat = np.array([
        [rows[m]['clean'] - rows[m][k] for k in ['fgsm','bim','pgd','cw']]
        for m in model_names])

    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(drop_mat, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=0.60)
    ax.set_xticks(range(4))
    ax.set_xticklabels(['FGSM (ε=0.1)','BIM (ε=0.1)','PGD (ε=0.1)','C&W (L2)'], fontsize=11)
    ax.set_yticks(range(len(model_names))); ax.set_yticklabels(model_names, fontsize=11)
    ax.set_title(f"Accuracy Drop Heatmap — {DATASET_NAME} (seed={seed})\n"
                 "Darker red = more vulnerable  |  Green = robust",
                 fontsize=12, fontweight='bold')
    for i in range(len(model_names)):
        for j in range(4):
            d = drop_mat[i,j]
            ax.text(j, i, f"−{d*100:.1f}%", ha='center', va='center',
                    fontsize=11, fontweight='bold', color='white' if d > 0.35 else 'black')
    plt.colorbar(im, ax=ax, label='Accuracy drop', fraction=0.03, pad=0.04)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/drop_heatmap_seed{seed}.png", dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: drop_heatmap_seed{seed}.png")

### CELL 20 | Training History — All 5 Models

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(28, 8))
fig.suptitle(f"Training History — {DATASET_NAME} | All 5 Models",
             fontsize=14, fontweight='bold')

std_models = [
    (hist_resnet,  'ResNet',           'steelblue'),
    (hist_fcn,     'FCN',              'darkorange'),
    (hist_lstm,    'LSTM',             'purple'),
    (hist_vanilla, 'LSTM-FCN Vanilla', 'seagreen'),
]

for col, (h, name, color) in enumerate(std_models):
    for row, (tr_k, val_k, label) in enumerate([
            ('loss','val_loss','Loss'),
            ('accuracy','val_accuracy','Acc')]):
        rtr  = h.get(tr_k,  [0])
        rval = h.get(val_k, [0])
        axes[row,col].plot(rtr,               color=color, lw=0.5, alpha=0.2)
        axes[row,col].plot(rval,              color=color, lw=0.5, alpha=0.2, ls='--')
        axes[row,col].plot(smooth_curve(rtr), color=color, lw=1.8, label='Train')
        axes[row,col].plot(smooth_curve(rval),color=color, lw=1.8, ls='--', label='Val')
        axes[row,col].set_title(f"{name} — {label}", fontsize=10)
        axes[row,col].legend(fontsize=7); axes[row,col].grid(alpha=0.3)

# CurAT column
adv_l = hist_robust.get('loss', [0])
adv_v = hist_robust.get('val_accuracy', [0])
axes[0,4].plot(adv_l,               color='crimson', lw=0.5, alpha=0.2)
axes[0,4].plot(smooth_curve(adv_l), color='crimson', lw=1.8)
axes[0,4].set_title("CurAT-LSTM-FCN — Loss", fontsize=10); axes[0,4].grid(alpha=0.3)
axes[1,4].plot(adv_v,               color='crimson', lw=0.5, alpha=0.2)
axes[1,4].plot(smooth_curve(adv_v), color='crimson', lw=1.8)
axes[1,4].set_title("CurAT-LSTM-FCN — Val Acc", fontsize=10); axes[1,4].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_history_seed{seed}.png", ...)
#plt.savefig(f"{SAVE_DIR}/training_history_all.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: training_history_all.png")



### CELL 21 | LSTM-FCN Vanilla vs CurAT-LSTM-FCN Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    f"LSTM-FCN Vanilla vs CurAT-LSTM-FCN — {DATASET_NAME}\n"
    "Same architecture · Different training strategy",
    fontsize=13, fontweight='bold')

v_clean = acc_matrix['LSTM-FCN Vanilla']['Clean']
r_clean = acc_matrix['CurAT-LSTM-FCN']['Clean']

for col, (aname, akey) in enumerate([('BIM','BIM'),('C&W','CW')]):
    ax   = axes[col]
    va   = acc_matrix['LSTM-FCN Vanilla'][akey]
    ra   = acc_matrix['CurAT-LSTM-FCN'][akey]
    gain = (v_clean - va) - (r_clean - ra)
    bars = ax.bar([0,1,2.6,3.6], [v_clean,va,r_clean,ra],
                  color=['#2980b9','#e74c3c','#27ae60','#f39c12'],
                  edgecolor='white', width=0.7, alpha=0.88)
    for bar, v in zip(bars, [v_clean,va,r_clean,ra]):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.005,
                f"{v*100:.2f}%", ha='center', fontsize=9, fontweight='bold')
    for x0,x1,y0,y1,c in [(0,1,v_clean,va,'#e74c3c'),(2.6,3.6,r_clean,ra,'#f39c12')]:
        ax.annotate('', xy=(x1,y1+0.01), xytext=(x0,y0-0.01),
                    arrowprops=dict(arrowstyle='->', color=c, lw=2.0))
        ax.text((x0+x1)/2+0.2,(y0+y1)/2,
                f"−{(y0-y1)*100:.2f}%", fontsize=9, color=c, fontweight='bold')
    y_b = min(va,ra)-0.04
    ax.annotate('', xy=(1,y_b), xytext=(3.6,y_b),
                arrowprops=dict(arrowstyle='<->', color='darkgreen', lw=1.8))
    ax.text(2.3, y_b-0.025, f"+{gain*100:.2f}% robustness gain",
            ha='center', fontsize=9, color='darkgreen', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#d5f5e3',
                      edgecolor='darkgreen', alpha=0.9))
    ax.set_xticks([0,1,2.6,3.6])
    ax.set_xticklabels(['Vanilla\nClean',f'Vanilla\n{aname}',
                        'CurAT\nClean',f'CurAT\n{aname}'], fontsize=10)
    ax.set_title(f"{aname} Attack (ε={EPSILON})", fontsize=12)
    ax.set_ylim(max(0, min(v_clean,va,r_clean,ra)-0.14), 1.08)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/vanilla_vs_curat_seed{seed}.png", ...)
# plt.savefig(f"{SAVE_DIR}/vanilla_vs_curat.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: vanilla_vs_curat.png")


### CELL 22 | Epsilon Sweep

In [ ]:
if seed == 0:   # sirf pehli baar chalao is dataset ke liye, baaki seeds mein skip
    eps_vals = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]
    sweep    = {m: {'fgsm':[], 'bim':[]} for m in model_names}
    print("Running epsilon sweep (seed=0 only — this is expensive, skipped for other seeds)...")
    for eps in eps_vals:
        Xf = fgsm_attack(resnet_model, X_test, y_test_cat, epsilon=eps)
        Xb = bim_attack( resnet_model, X_test, y_test_cat, epsilon=eps, num_iter=100)
        for mname, model in all_models.items():
            sweep[mname]['fgsm'].append(accuracy_score(y_test_int, np.argmax(model.predict(Xf,verbose=0),axis=1)))
            sweep[mname]['bim'].append(accuracy_score(y_test_int, np.argmax(model.predict(Xb,verbose=0),axis=1)))
        print(f"  ε={eps:.2f} done")

    mc = {'ResNet':'steelblue','FCN':'darkorange','LSTM':'purple','LSTM-FCN Vanilla':'seagreen','CurAT-LSTM-FCN':'crimson'}
    ms = {'ResNet':'-','FCN':'--','LSTM':'-.','LSTM-FCN Vanilla':':','CurAT-LSTM-FCN':'-'}

    fig, (ax1,ax2) = plt.subplots(1,2, figsize=(15,5), sharey=True)
    fig.suptitle(f"Accuracy vs ε — {DATASET_NAME} (seed=0)", fontsize=13, fontweight='bold')
    for mname in model_names:
        row = get_row(mname)
        c = row['clean'] if row else 0.0
        kw = dict(color=mc[mname], linestyle=ms[mname], linewidth=2.0, label=f"{mname} ({c*100:.1f}%)")
        ax1.plot(eps_vals, sweep[mname]['fgsm'], 'o', **kw)
        ax2.plot(eps_vals, sweep[mname]['bim'],  's', **kw)
    for ax, title in [(ax1,'FGSM'),(ax2,'BIM (100 iter)')]:
        ax.set_xlabel("ε"); ax.set_ylabel("Accuracy")
        ax.set_title(title); ax.legend(fontsize=8)
        ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/epsilon_sweep_seed0.png", dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: epsilon_sweep_seed0.png")
else:
    print(f"⚠ Epsilon sweep skipped for seed={seed} (only run once, at seed=0, to save GPU time)")

### CELL 23 | Final Summary + Download All Figures

In [ ]:
SEP = "="*95
print(f"\n{SEP}")
print(f"  FINAL SUMMARY — {DATASET_NAME}  (seed={seed})")
print(SEP)
print(f"  {'Model':<22} {'Clean':>8} {'FGSM':>8} {'BIM':>8} {'PGD':>8} {'C&W':>8}  Mode")
print("  " + "-"*78)
for mname in model_names:
    row = get_row(mname)
    if row is None:
        print(f"  {mname:<22}  [no data for seed={seed}]")
        continue
    print(f"  {mname:<22} "
          f"{row['clean']*100:>7.2f}% {row['fgsm']*100:>7.2f}% {row['bim']*100:>7.2f}% "
          f"{row['pgd']*100:>7.2f}% {row['cw']*100:>7.2f}%  {attack_mode[mname]}")
print(SEP)

summary = {
    'dataset': DATASET_NAME,
    'seed': seed,
    'epsilon': EPSILON,
    'results': {mname: {k: round(get_row(mname)[k], 4) for k in ['clean','fgsm','bim','pgd','cw']}
                for mname in model_names if get_row(mname) is not None}
}
with open(f"{CKPT_DIR}/final_results_seed{seed}.json", 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\n✓ Results JSON saved to Drive: final_results_seed{seed}.json")

import shutil
from google.colab import files
zip_path = f"/content/{DATASET_NAME}_seed{seed}_results"
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
print(f"✓ Created {zip_path}.zip — downloading now...")
files.download(f"{zip_path}.zip")
print(f"\n  ★ Next: change seed (0→1→2→3→4) in Cell 2's SEED, then re-run Cell 2 onward.")
print(f"  ★ After all 5 seeds, change DATASET_NAME in Cell 2 for the next dataset (seed resets to 0). ★")